### Imports

In [ ]:
import numpy as np
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_curve, auc, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from transformers import ViTMAEForPreTraining, AutoImageProcessor
from torch.utils.data import Dataset, DataLoader
import os
from PIL import Image
import torch.nn.functional as F

### Device Check

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")   # Apple GPU
    print("Using MPS (Apple GPU)")
elif torch.cuda.is_available():
    device = torch.device("cuda")  # NVIDIA GPU
    print("Using CUDA")
else:
    device = torch.device("cpu")   # fallback
    print("Using CPU")

### Loading MAE Encoder

In [ ]:
model_path = "/home/le.song/mae_fidd_model"
model = ViTMAEForPreTraining.from_pretrained(model_path).to(device)

### Image Processor

In [ ]:
processor = AutoImageProcessor.from_pretrained("facebook/vit-mae-base")

### Dataset Definition

In [ ]:
class UAD_Dataset(Dataset):
    def __init__(self, root_dir, processor, split=None, return_path=False):
        """
        Args:
            root_dir: Base directory (e.g., /home/le.song/Unsupervised_AD_Data/Real)
            processor: ViTMAEImageProcessor
            split: 'val' or 'test' (appended to root_dir)
            return_path: If True, __getitem__ returns (pixels, label, path)
        """
        self.target_dir = os.path.join(root_dir, split) if split else root_dir
        self.processor = processor
        self.return_path = return_path
        self.image_data = [] 

        if not os.path.exists(self.target_dir):
            raise FileNotFoundError(f"Directory not found: {self.target_dir}")

        valid_extensions = (".png", ".jpg", ".jpeg")
        for f in os.listdir(self.target_dir):
            if f.lower().endswith(valid_extensions):
                full_path = os.path.join(self.target_dir, f)
                # Ground Truth Labeling for supervised evaluation(Only training is unsupervised)
                label = 1 if "fake" in f.lower() else 0
                self.image_data.append((full_path, label))
        
        # Performance Telemetry
        real_count = sum(1 for _, l in self.image_data if l == 0)
        fake_count = sum(1 for _, l in self.image_data if l == 1)
        print(f"--- [{split if split else 'Root'}] Dataset Loaded ---")
        print(f"Total: {len(self.image_data)} | Real: {real_count} | Fake: {fake_count}")

    def __len__(self):
        return len(self.image_data)

    def __getitem__(self, idx):
        img_path, label = self.image_data[idx]
        try:
            img = Image.open(img_path).convert("RGB")
            pixel_values = self.processor(images=img, return_tensors="pt")["pixel_values"].squeeze(0)
            
            if self.return_path:
                return pixel_values, label, img_path
            return pixel_values, label
            
        except Exception as e:
            print(f"Skip error on {img_path}: {e}")
            return torch.zeros((3, 224, 224)), label

### Data Loaders

In [ ]:
val_dir = "/home/le.song/Unsupervised_AD_Data/Real"
test_dir = "/home/le.song/Unsupervised_AD_Data/RF"

val_dataset = UAD_Dataset(root_dir=val_dir, processor=processor, split='val')
test_dataset = UAD_Dataset(root_dir=test_dir, processor=processor, split='test')
    
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

### Calibrate Function Against Valset

In [ ]:
def calibrate(model, dataloader, device):
    """
    Finds the 'Normal' distribution of reconstruction errors using real images.
    Returns: mu (mean), std (standard deviation), and the 1-sigma threshold.
    """
    model.eval()
    mse_scores = []
    
    print(f"--- Starting Calibration on {len(dataloader.dataset)} Real Images ---")
    with torch.no_grad():
        for pixels, _ in dataloader:
            pixels = pixels.to(device)
            # MAE forward pass calculates reconstruction MSE internally
            outputs = model(pixel_values=pixels)
            mse_scores.append(outputs.loss.item())
            
    mu = np.mean(mse_scores)
    std = np.std(mse_scores)
    threshold = mu + (1 * std)
    
    print(f"Calibration Done. Threshold (1-sigma): {threshold:.6f}")
    return mu, std, threshold

### Evaluate Function Using Testset

In [ ]:
def evaluate(model, dataloader, threshold, device):
    model.eval()
    y_true, y_pred, y_scores = [], [], []
    
    print(f"--- Evaluating {len(dataloader.dataset)} Mixed Images ---")
    with torch.no_grad():
        for pixels, labels in dataloader:
            pixels = pixels.to(device)
            outputs = model(pixel_values=pixels)
            mse = outputs.loss.item()
            
            prediction = 1 if mse > threshold else 0
            
            y_true.append(labels.item())
            y_pred.append(prediction)
            y_scores.append(mse) # Raw scores for ROC/AUC
            
    # --- Metric Calculations ---
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')
    fpr_list, tpr_list, _ = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr_list, tpr_list)
    cm = confusion_matrix(y_true, y_pred)

    # --- Printing Results ---
    print("\n" + "="*45)
    print("      DETECTION PERFORMANCE SUMMARY")
    print("="*45)
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"ROC AUC:   {roc_auc:.4f}")
    print("\nDetailed Classification Report:")
    print(classification_report(y_true, y_pred, target_names=["Real", "Fake"]))

    # --- Plotting ROC Curve ---
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(fpr_list, tpr_list, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC)')
    plt.legend(loc="lower right")

    # --- Plotting Confusion Matrix ---
    plt.subplot(1, 2, 2)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Real", "Fake"])
    disp.plot(ax=plt.gca(), cmap=plt.cm.Blues, colorbar=False)
    plt.title('Confusion Matrix')
    
    plt.tight_layout()
    plt.show()

    return {
        "y_true": y_true, "y_pred": y_pred, "y_scores": y_scores,
        "metrics": {"acc": acc, "prec": precision, "rec": recall, "f1": f1, "auc": roc_auc}
    }

### Heatmaps for Reconstruction Error

In [ ]:
# def generate_heatmaps(model, dataloader, device, num_images=10):

#     os.makedirs("heatmaps_raw", exist_ok=True)
#     print(f"\n[3/3] Saving {num_images} Patch-Error Grids to 'heatmaps_raw/'...")
    
#     count = 0
#     with torch.no_grad():
#         for pixels, labels, paths in dataloader:
#             if count >= num_images: break
            
#             # 1. Forward Pass
#             pixels = pixels.to(device)
#             outputs = model(pixel_values=pixels)
            
#             # 2. Calculate Squared Error per patch
#             target = model.patchify(pixels)
#             # (Batch, 196, 768) -> Mean over pixel dimension -> (1, 196)
#             patch_loss = ((outputs.logits - target) ** 2).mean(dim=-1)
            
#             # 3. Reshape to the physical 14x14 grid
#             heatmap = patch_loss.view(14, 14).cpu().numpy()
            
#             # 4. Normalize to 0-255 for a standard grayscale image
#             heatmap_norm = ((heatmap - heatmap.min()) / (heatmap.max() - heatmap.min()) * 255).astype(np.uint8)
            
#             # 5. Save using the original filename (e.g., map_ILSVRC_001.png) 
#             # to know which fake image it corresponds to for future overlays
#             orig_filename = os.path.basename(paths[0]).split('.')[0]
#             save_path = f"heatmaps_raw/map_{orig_filename}.png"
            
#             Image.fromarray(heatmap_norm).save(save_path)
            
#             count += 1

#     print(f"Maps saved.")

### Heatmaps for Reconstruction Error(Automatic Overlay)

In [ ]:
def generate_heatmaps(model, dataloader, device, num_real=5, num_fake=5, save_prefix="UAD"):
    model.eval()
    os.makedirs("heatmaps", exist_ok=True)

    print(f"--- Generating {num_real} Real and {num_fake} Fake Heatmaps ---")

    count_real = 0
    count_fake = 0
    
    with torch.no_grad():
        for pixels, labels, paths in dataloader:
            is_fake = labels[0].item() == 1
            
            # Skip if we already have enough of this class
            if is_fake and count_fake >= num_fake: continue
            if not is_fake and count_real >= num_real: continue

            pixels = pixels.to(device)
            outputs = model(pixel_values=pixels)
            logits = outputs.logits 

            # 1. Calculate patch-wise error
            target = model.patchify(pixels)
            patch_loss = (logits - target) ** 2
            patch_loss = patch_loss.mean(dim=-1) # (1, 196)

            # 2. Reshape to 14x14 grid
            heatmap = patch_loss.view(14, 14).cpu().numpy()

            # 3. Upsample heatmap to 224x224
            heatmap_resized = np.array(Image.fromarray(heatmap).resize((224, 224), resample=Image.BICUBIC))

            # 4. Plotting
            fig, axes = plt.subplots(1, 3, figsize=(15, 5))

            # Original Image (Denormalize)
            orig = pixels.squeeze().cpu().permute(1, 2, 0).numpy()
            orig = (orig - orig.min()) / (orig.max() - orig.min())

            # Reconstruction
            recon = model.unpatchify(logits).squeeze().cpu().permute(1, 2, 0).numpy()
            recon = (recon - recon.min()) / (recon.max() - recon.min())

            label_text = "Fake" if is_fake else "Real"
            
            axes[0].imshow(orig)
            axes[0].set_title(f"Original ({label_text})")
            axes[0].axis('off')

            axes[1].imshow(recon)
            axes[1].set_title("MAE Reconstruction")
            axes[1].axis('off')

            # Heatmap Overlay
            # We use a consistent vmax so that 'red' means the same intensity across images
            axes[2].imshow(orig) 
            im = axes[2].imshow(heatmap_resized, cmap='jet', alpha=0.6, vmin=0, vmax=0.1) 
            axes[2].set_title(f"Error Heatmap (MSE: {patch_loss.mean().item():.4f})")
            axes[2].axis('off')

            plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

            # 5. Save the result
            clean_name = os.path.basename(paths[0]).split('.')[0]
            plt.savefig(f"heatmaps/{save_prefix}_{label_text}_{clean_name}.png", bbox_inches='tight')
            plt.close()

            if is_fake: count_fake += 1
            else: count_real += 1
            
            if count_real >= num_real and count_fake >= num_fake: break

    print(f"Comparison heatmaps saved in 'heatmaps/'")

### Execution Block

In [ ]:
if __name__ == "__main__":
    mu, std, threshold = calibrate(model, val_loader, device)
    results = evaluate(model, test_loader, threshold, device)

    y_scores = results["y_scores"]
    y_true = results["y_true"]
    metrics = results["metrics"]

    plt.figure(figsize=(10, 6))
    

    real_scores = [s for s, l in zip(y_scores, y_true) if l == 0]
    fake_scores = [s for s, l in zip(y_scores, y_true) if l == 1]
    #Error Distribution Plot
    plt.hist(real_scores, bins=50, alpha=0.5, label='Real (COCO)', color='blue', density=True)
    plt.hist(fake_scores, bins=50, alpha=0.5, label='Fake (Gen-AI)', color='red', density=True)
    plt.axvline(threshold, color='green', linestyle='--', label=f'Threshold (2σ: {threshold:.4f})')
    plt.title("Reconstruction Error Distribution: Real vs. Fake")
    plt.xlabel("MSE (Anomaly Score)")
    plt.ylabel("Density")
    plt.legend()
    plt.savefig('error_distribution.png')
    plt.show()

    #Heatmap Generation
    viz_dataset = UAD_Dataset(root_dir=test_dir, processor=processor, split='test', return_path=True)
    viz_loader = DataLoader(viz_dataset, batch_size=1, shuffle=True)
    
    # 'Now generates overlays for both real and fake images'
    generate_heatmaps(model, viz_loader, device, num_real=5, num_fake=5)

    # Data Storage
    with open("experiment_summary.txt", "w") as f:
        f.write("MAE UNSUPERVISED DETECTION SUMMARY\n")
        f.write("================================\n")
        f.write(f"Calibration Mu:  {mu:.6f}\n")
        f.write(f"Calibration Std: {std:.6f}\n")
        f.write(f"Final Threshold: {threshold:.6f}\n")
        f.write(f"Test Accuracy:   {metrics['acc']:.4f}\n")
        f.write(f"Test F1-Score:   {metrics['f1']:.4f}\n")
        f.write(f"Test AUC:        {metrics['auc']:.4f}\n")
        f.write(f"Test Precision:  {metrics['prec']:.4f}\n")
        f.write(f"Test Recall:     {metrics['rec']:.4f}\n")
    
    print("Distribution plot saved as error_distribution.png")
    print("Experiment summary saved to experiment_summary.txt")